# 🏠 Project 002 — Dutch Housing Market Forecast
**Portfolio project | Ishan Sewnandan**

Regional price trend modelling using CBS & Kadaster open data.  
Tools: Python · Pandas · Scikit-learn · Plotly

---
### Doel
- Woningprijsontwikkeling per provincie analyseren (2015–2024)
- Regressiemodel trainen om prijstrends te voorspellen
- Interactieve visualisaties bouwen voor de portfolio

### Databronnen
- **CBS StatLine** — `83625NED`: Bestaande koopwoningen; gemiddelde verkoopprijzen, regio
- **CBS StatLine** — `85773NED`: Prijsindex bestaande koopwoningen (2020=100)


## 0. Setup & Installatie

In [ ]:
# Installeer benodigde packages (alleen eerste keer)
# !pip install cbsodata pandas numpy scikit-learn plotly nbformat

import cbsodata
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

print('✅ Alle packages geladen')

## 1. Data Ophalen via CBS Open Data API

In [ ]:
# ─── Dataset 1: Gemiddelde verkoopprijzen per regio ───────────────────────────
print('📥 Dataset 1 ophalen: gemiddelde verkoopprijzen...')
df_raw = pd.DataFrame(cbsodata.get_data('83625NED'))
print(f'   → {len(df_raw):,} rijen, {df_raw.shape[1]} kolommen')
df_raw.head(3)

In [ ]:
# Bekijk alle kolomnamen
print('Kolommen:')
for col in df_raw.columns:
    print(f'  {col}: {df_raw[col].dtype} | voorbeeld: {df_raw[col].iloc[0]}')

In [ ]:
# ─── Dataset 2: Prijsindex (2020=100) ────────────────────────────────────────
print('📥 Dataset 2 ophalen: prijsindex bestaande koopwoningen...')
df_index_raw = pd.DataFrame(cbsodata.get_data('85773NED'))
print(f'   → {len(df_index_raw):,} rijen, {df_index_raw.shape[1]} kolommen')
df_index_raw.head(3)

## 2. Data Cleaning & Transformatie

In [ ]:
df = df_raw.copy()

# ─── Kolomnamen normaliseren ─────────────────────────────────────────────────
df.columns = [c.strip() for c in df.columns]
print('Kolomnamen:', list(df.columns))

In [ ]:
# ─── Periode kolom parsen ─────────────────────────────────────────────────────
# CBS gebruikt formaat: '2023JJ00' (jaarlijks) of '2023KW01' (kwartaal)
# We filteren op jaarlijkse waarden

periode_col = 'Perioden'  # pas aan als kolomnaam anders is
regio_col   = 'RegioS'    # pas aan als kolomnaam anders is

# Selecteer jaarlijkse perioden
df_jaar = df[df[periode_col].str.contains('JJ00', na=False)].copy()
df_jaar['jaar'] = df_jaar[periode_col].str[:4].astype(int)

# Filter op relevante jaren
df_jaar = df_jaar[(df_jaar['jaar'] >= 2015) & (df_jaar['jaar'] <= 2024)]

print(f'Jaarlijkse rijen (2015–2024): {len(df_jaar):,}')
print(f'Unieke regio\'s: {df_jaar[regio_col].nunique()}')
print('\nRegio overzicht:')
print(df_jaar[regio_col].unique())

In [ ]:
# ─── Prijskolom identificeren & cleanen ──────────────────────────────────────
# Zoek de kolom met gemiddelde verkoopprijs
prijs_cols = [c for c in df_jaar.columns if 'prijs' in c.lower() or 'Prijs' in c or 'euro' in c.lower()]
print('Mogelijke prijskolommen:', prijs_cols)

# Gebruik eerste gevonden prijskolom (aanpassen indien nodig)
prijs_col = prijs_cols[0] if prijs_cols else None
print(f'\nGekozen kolom: {prijs_col}')

if prijs_col:
    df_jaar['gem_prijs'] = pd.to_numeric(df_jaar[prijs_col], errors='coerce')
    print(f'\nPrijsrange: €{df_jaar["gem_prijs"].min():,.0f} – €{df_jaar["gem_prijs"].max():,.0f}')
    print(f'Missing values: {df_jaar["gem_prijs"].isna().sum()}')

In [ ]:
# ─── Provincies filteren ──────────────────────────────────────────────────────
# CBS codering: provincies starten met 'PV' of zijn herkenbaar aan naam
provincies = [
    'Groningen', 'Friesland', 'Drenthe', 'Overijssel', 'Flevoland',
    'Gelderland', 'Utrecht', 'Noord-Holland', 'Zuid-Holland', 'Zeeland',
    'Noord-Brabant', 'Limburg'
]

# Probeer te filteren op provincienamen
df_prov = df_jaar[df_jaar[regio_col].str.strip().isin(provincies)].copy()

# Als dat leeg is, gebruik alle regio's
if len(df_prov) == 0:
    print('⚠️  Geen provincies gevonden op naam. Alle regio\'s worden gebruikt.')
    df_prov = df_jaar.copy()
else:
    print(f'✅ {df_prov[regio_col].nunique()} provincies gevonden')

df_prov['regio'] = df_prov[regio_col].str.strip()
df_prov = df_prov.dropna(subset=['gem_prijs'])
print(f'Schone dataset: {len(df_prov):,} rijen')
df_prov[['regio', 'jaar', 'gem_prijs']].head(10)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ─── Statistieken per regio ───────────────────────────────────────────────────
stats = df_prov.groupby('regio')['gem_prijs'].agg(
    gemiddeld='mean',
    min_prijs='min',
    max_prijs='max',
    stijging=lambda x: ((x.iloc[-1] / x.iloc[0]) - 1) * 100 if len(x) > 1 else 0
).round(0).sort_values('gemiddeld', ascending=False)

print('📊 Gemiddelde verkoopprijs per regio (2015–2024):')
stats.style.format({'gemiddeld': '€{:,.0f}', 'min_prijs': '€{:,.0f}', 
                    'max_prijs': '€{:,.0f}', 'stijging': '{:.1f}%'})

In [ ]:
# ─── Plot 1: Prijsontwikkeling per provincie ──────────────────────────────────
fig1 = px.line(
    df_prov.sort_values('jaar'),
    x='jaar', y='gem_prijs', color='regio',
    title='🏠 Gemiddelde Verkoopprijs Woningen per Regio (2015–2024)',
    labels={'gem_prijs': 'Gemiddelde Prijs (€)', 'jaar': 'Jaar', 'regio': 'Regio'},
    template='plotly_white',
    line_shape='spline'
)
fig1.update_layout(
    font_family='Georgia',
    title_font_size=18,
    hovermode='x unified',
    yaxis_tickformat='€,.0f',
    legend=dict(orientation='v', x=1.02, y=1)
)
fig1.update_traces(line_width=2.5)
fig1.show()
fig1.write_html('housing_trends.html')
print('✅ Opgeslagen als housing_trends.html')

In [ ]:
# ─── Plot 2: Prijsstijging % per regio (bar chart) ────────────────────────────
stijging_df = df_prov.groupby('regio').apply(
    lambda g: pd.Series({
        'prijs_2015': g[g['jaar'] == g['jaar'].min()]['gem_prijs'].values[0] if len(g[g['jaar'] == g['jaar'].min()]) > 0 else np.nan,
        'prijs_2024': g[g['jaar'] == g['jaar'].max()]['gem_prijs'].values[0] if len(g[g['jaar'] == g['jaar'].max()]) > 0 else np.nan,
    })
).dropna()

stijging_df['stijging_pct'] = ((stijging_df['prijs_2024'] / stijging_df['prijs_2015']) - 1) * 100
stijging_df = stijging_df.sort_values('stijging_pct', ascending=True).reset_index()

fig2 = px.bar(
    stijging_df, x='stijging_pct', y='regio',
    orientation='h',
    title='📈 Totale Prijsstijging per Regio (eerste jaar → laatste jaar)',
    labels={'stijging_pct': 'Prijsstijging (%)', 'regio': 'Regio'},
    color='stijging_pct',
    color_continuous_scale='RdYlGn',
    template='plotly_white'
)
fig2.update_layout(font_family='Georgia', title_font_size=16, showlegend=False)
fig2.update_traces(texttemplate='%{x:.1f}%', textposition='outside')
fig2.show()

## 4. Feature Engineering

In [ ]:
# ─── Features aanmaken ────────────────────────────────────────────────────────
df_model = df_prov[['regio', 'jaar', 'gem_prijs']].copy().dropna()

# Label encoding voor regio
le = LabelEncoder()
df_model['regio_encoded'] = le.fit_transform(df_model['regio'])

# Jaar features
df_model['jaar_norm'] = df_model['jaar'] - df_model['jaar'].min()  # 0-based
df_model['jaar_kwadraat'] = df_model['jaar_norm'] ** 2             # niet-lineaire trend

# Lagged prijzen per regio (vorig jaar)
df_model = df_model.sort_values(['regio', 'jaar'])
df_model['prijs_lag1'] = df_model.groupby('regio')['gem_prijs'].shift(1)
df_model['prijs_lag2'] = df_model.groupby('regio')['gem_prijs'].shift(2)

# YoY groei
df_model['yoy_groei'] = df_model.groupby('regio')['gem_prijs'].pct_change() * 100

# Rolling gemiddelde (3 jaar)
df_model['prijs_ma3'] = df_model.groupby('regio')['gem_prijs'].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)

# Verwijder rijen met NaN (door lag)
df_model_clean = df_model.dropna()
print(f'Model dataset: {len(df_model_clean)} rijen')
print(f'Features: regio_encoded, jaar_norm, jaar_kwadraat, prijs_lag1, prijs_lag2, yoy_groei, prijs_ma3')
df_model_clean.head()

## 5. Model Training & Evaluatie

In [ ]:
# ─── Train/test split ─────────────────────────────────────────────────────────
feature_cols = ['regio_encoded', 'jaar_norm', 'jaar_kwadraat', 
                'prijs_lag1', 'prijs_lag2', 'yoy_groei', 'prijs_ma3']
target_col = 'gem_prijs'

X = df_model_clean[feature_cols]
y = df_model_clean[target_col]

# Tijdsgebaseerde split: train op 2015-2021, test op 2022-2024
train_mask = df_model_clean['jaar'] <= 2021
X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

print(f'Train: {len(X_train)} rijen ({df_model_clean[train_mask]["jaar"].min()}–{df_model_clean[train_mask]["jaar"].max()})')
print(f'Test:  {len(X_test)} rijen ({df_model_clean[~train_mask]["jaar"].min()}–{df_model_clean[~train_mask]["jaar"].max()})')

In [ ]:
# ─── Modellen trainen ─────────────────────────────────────────────────────────
modellen = {
    'Linear Regression': LinearRegression(),
    'Random Forest':     RandomForestRegressor(n_estimators=200, random_state=42, max_depth=8),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, random_state=42, learning_rate=0.05)
}

resultaten = {}

for naam, model in modellen.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    
    resultaten[naam] = {'model': model, 'y_pred': y_pred, 'MAE': mae, 'RMSE': rmse, 'R²': r2}
    print(f'\n{naam}')
    print(f'  MAE:  €{mae:>10,.0f}')
    print(f'  RMSE: €{rmse:>10,.0f}')
    print(f'  R²:   {r2:>10.4f}')

In [ ]:
# ─── Beste model kiezen ───────────────────────────────────────────────────────
beste_naam = max(resultaten, key=lambda k: resultaten[k]['R²'])
beste_model = resultaten[beste_naam]['model']
print(f'🏆 Beste model: {beste_naam} (R² = {resultaten[beste_naam]["R²"]:.4f})')

In [ ]:
# ─── Plot 3: Actual vs Predicted ──────────────────────────────────────────────
y_pred_best = resultaten[beste_naam]['y_pred']
test_df = df_model_clean[~train_mask].copy()
test_df['voorspeld'] = y_pred_best

fig3 = go.Figure()
for regio in test_df['regio'].unique():
    sub = test_df[test_df['regio'] == regio].sort_values('jaar')
    fig3.add_trace(go.Scatter(x=sub['gem_prijs'], y=sub['voorspeld'],
                              mode='markers', name=regio, 
                              marker=dict(size=10, opacity=0.8)))

# Perfecte voorspellijn
minv, maxv = y_test.min(), y_test.max()
fig3.add_trace(go.Scatter(x=[minv, maxv], y=[minv, maxv],
                          mode='lines', name='Perfecte voorspelling',
                          line=dict(color='black', dash='dash', width=1.5)))

fig3.update_layout(
    title=f'🎯 Actual vs Predicted — {beste_naam} (R² = {resultaten[beste_naam]["R²"]:.3f})',
    xaxis_title='Werkelijke prijs (€)', yaxis_title='Voorspelde prijs (€)',
    template='plotly_white', font_family='Georgia',
    xaxis_tickformat='€,.0f', yaxis_tickformat='€,.0f'
)
fig3.show()

In [ ]:
# ─── Feature importance (Random Forest / GB) ─────────────────────────────────
if hasattr(beste_model, 'feature_importances_'):
    fi_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': beste_model.feature_importances_
    }).sort_values('importance', ascending=True)

    fig4 = px.bar(fi_df, x='importance', y='feature', orientation='h',
                  title=f'🔍 Feature Importance — {beste_naam}',
                  color='importance', color_continuous_scale='Blues',
                  template='plotly_white')
    fig4.update_layout(font_family='Georgia', showlegend=False)
    fig4.show()

## 6. Voorspelling 2025–2027

In [ ]:
# ─── Forecast naar toekomst ───────────────────────────────────────────────────
forecast_jaren = [2025, 2026, 2027]
forecast_rows = []

for regio in df_model_clean['regio'].unique():
    regio_data = df_model_clean[df_model_clean['regio'] == regio].sort_values('jaar')
    
    if len(regio_data) < 2:
        continue
    
    # Laatste bekende waarden
    laatste = regio_data.iloc[-1]
    op_een_na = regio_data.iloc[-2]
    
    prijs_current = laatste['gem_prijs']
    prijs_prev    = op_een_na['gem_prijs']
    jaar_min      = df_model_clean['jaar'].min()
    
    for jaar in forecast_jaren:
        yoy = ((prijs_current / prijs_prev) - 1) * 100 if prijs_prev > 0 else 0
        ma3 = regio_data['gem_prijs'].tail(3).mean()
        
        rij = {
            'regio': regio,
            'regio_encoded': le.transform([regio])[0],
            'jaar': jaar,
            'jaar_norm': jaar - jaar_min,
            'jaar_kwadraat': (jaar - jaar_min) ** 2,
            'prijs_lag1': prijs_current,
            'prijs_lag2': prijs_prev,
            'yoy_groei': yoy,
            'prijs_ma3': ma3
        }
        
        pred = beste_model.predict(pd.DataFrame([rij])[feature_cols])[0]
        rij['gem_prijs'] = pred
        rij['type'] = 'Forecast'
        forecast_rows.append(rij)
        
        # Schuif waarden op voor volgend jaar
        prijs_prev    = prijs_current
        prijs_current = pred

df_forecast = pd.DataFrame(forecast_rows)
print(f'✅ Forecast klaar: {len(df_forecast)} rijen voor {forecast_jaren}')
df_forecast[['regio', 'jaar', 'gem_prijs']].head()

In [ ]:
# ─── Plot 5: Historisch + Forecast per regio ──────────────────────────────────
df_hist = df_model_clean[['regio', 'jaar', 'gem_prijs']].copy()
df_hist['type'] = 'Historisch'

df_combined = pd.concat([df_hist, df_forecast[['regio', 'jaar', 'gem_prijs', 'type']]], ignore_index=True)
df_combined = df_combined.sort_values(['regio', 'jaar'])

fig5 = px.line(
    df_combined, x='jaar', y='gem_prijs', color='regio',
    line_dash='type',
    title='🔮 Woningprijzen: Historisch + Forecast 2025–2027',
    labels={'gem_prijs': 'Gemiddelde Prijs (€)', 'jaar': 'Jaar'},
    template='plotly_white',
    line_shape='spline'
)

# Verticale lijn op grens historisch/forecast
fig5.add_vline(x=2024.5, line_dash='dot', line_color='gray', 
               annotation_text='Forecast →', annotation_position='top right')

fig5.update_layout(
    font_family='Georgia',
    title_font_size=18,
    yaxis_tickformat='€,.0f',
    hovermode='x unified',
    height=550
)
fig5.show()
fig5.write_html('housing_forecast.html')
print('✅ Opgeslagen als housing_forecast.html')

## 7. Model Performance Samenvatting

In [ ]:
# ─── Vergelijkingstabel alle modellen ────────────────────────────────────────
perf_df = pd.DataFrame([
    {'Model': naam, 'MAE (€)': v['MAE'], 'RMSE (€)': v['RMSE'], 'R²': v['R²']}
    for naam, v in resultaten.items()
]).set_index('Model').sort_values('R²', ascending=False)

print('📊 Model Performance Vergelijking:')
perf_df.style.format({
    'MAE (€)': '€{:,.0f}',
    'RMSE (€)': '€{:,.0f}',
    'R²': '{:.4f}'
}).background_gradient(subset=['R²'], cmap='Greens')

In [ ]:
# ─── Business conclusies ─────────────────────────────────────────────────────
print('='*60)
print('📋 PROJECT 002 — CONCLUSIES')
print('='*60)
print(f'\n🏆 Beste model: {beste_naam}')
print(f'   R²:   {resultaten[beste_naam]["R²"]:.4f}')
print(f'   MAE:  €{resultaten[beste_naam]["MAE"]:,.0f} gemiddelde fout')

if len(stijging_df) > 0:
    duurste = stijging_df.nlargest(1, 'stijging_pct').iloc[0]
    goedkoopste = stijging_df.nsmallest(1, 'stijging_pct').iloc[0]
    print(f'\n📈 Hoogste stijging: {duurste["regio"]} (+{duurste["stijging_pct"]:.1f}%)')
    print(f'📉 Laagste stijging: {goedkoopste["regio"]} (+{goedkoopste["stijging_pct"]:.1f}%)')

print('\n✅ Outputs gegenereerd:')
print('   - housing_trends.html    → prijstrends per regio')
print('   - housing_forecast.html  → forecast 2025-2027')
print('\n🚀 Klaar voor portfolio!')

---
## 📁 Gegenereerde bestanden

| Bestand | Beschrijving |
|---|---|
| `housing_trends.html` | Interactieve prijstrends per regio (Plotly) |
| `housing_forecast.html` | Forecast 2025–2027 + historisch (Plotly) |

## 🔗 Databronnen
- [CBS StatLine 83625NED](https://opendata.cbs.nl/statline/#/CBS/nl/dataset/83625NED) — Gemiddelde verkoopprijzen bestaande koopwoningen
- [CBS StatLine 85773NED](https://www.cbs.nl/nl-nl/cijfers/detail/85773NED) — Prijsindex bestaande koopwoningen

---
*Project 002 | Ishan Sewnandan | Rotterdam, 2025*